# Music Annotation Project — Notebook 84
## A5 A1-Based Safe Annotation Cleanup Evaluation

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook evaluates one completed, locked **A5** run. It does not fit, tune, recalibrate or rerun A5.

### Comparators
- **V8.9**: historical frozen comparator.
- **A1**: primary baseline; every AMBIGUOUS pixel retains the exact observed byte.
- **A3**: research comparator; AMBIGUOUS pixels with `p_print >= 0.50` retain observed, with the remainder retaining V8.9 treatment.
- **A5**: A1 plus a frozen `SAFE_ANNOTATION_CLEANUP` decision.

### Required A5 run structure
```text
prediction_lock.json
runtime_roster.csv
pages/<page_key>/
  observed.png  seed.png  authority.png  ambiguous.png
  v8_9.png  a1.png  a3.png  a5.png  a5_cleanup_mask.png
```

### Frozen provisional gates
- Exact print-preservation loss versus A1: lower 95% page-bootstrap bound no worse than **-0.25 pp**.
- Material-print-worsening increase versus A1: harmful upper 95% bound no greater than **+0.25 pp**.
- Annotation material-improvement gain: mean at least **+0.25 pp** and lower 95% bound above zero.
- Zero 8-connected A5 false-cleanup components containing **20+ truth-PRINT pixels**.
- Zero change outside A5 cleanup, zero change outside authority, and fail-closed A1 fallback.

A passing verdict permits freezing a challenger only. It does not authorise production promotion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib seaborn scikit-image

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, gc
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
A5_ROOT=ROOT/'a5_safe_annotation_cleanup'
OUTPUT_ROOT=ROOT/'a5_safe_annotation_cleanup_evaluation'
A5_RUN_OVERRIDE=None
DARK_THRESHOLD=200; MATERIAL_MARGIN=5; PRINT_TOLERANCE_RADIUS=1
MAX_EXACT_PRINT_PRESERVATION_LOSS_PP=.25
MAX_MATERIAL_PRINT_WORSENING_INCREASE_PP=.25
MIN_ANNOTATION_IMPROVEMENT_GAIN_PP=.25
CATASTROPHIC_PRINT_COMPONENT_PIXELS=20
BOOTSTRAP_RESAMPLES=10_000; BOOTSTRAP_SEED=84005
for p in [A5_ROOT,OUTPUT_ROOT]: p.mkdir(parents=True,exist_ok=True)

def require(x,msg):
    if not x: raise RuntimeError(msg)
def read_gray(p):
    p=Path(p); require(p.exists(),f'Missing {p}')
    x=cv2.imread(str(p),0); require(x is not None,f'Unreadable {p}'); return x
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def latest_complete(root):
    q=[p for p in root.iterdir() if p.is_dir() and (p/'prediction_lock.json').exists() and (p/'runtime_roster.csv').exists() and (p/'pages').is_dir()] if root.exists() else []
    return max(q,key=lambda p:p.stat().st_mtime) if q else None

## 1. Resolve the locked A5 run and freeze the evaluation identity

In [ ]:
A5_RUN=Path(A5_RUN_OVERRIDE) if A5_RUN_OVERRIDE else latest_complete(A5_ROOT)
require(A5_RUN is not None,'No complete locked A5 run found.')
lock=json.loads((A5_RUN/'prediction_lock.json').read_text())
require(lock.get('state') in {'CLOSED_BEFORE_CLEAN_TRUTH','LOCKED_BEFORE_TRUTH','CLOSED'},f'Prediction lock not closed: {lock.get("state")}')
roster=pd.read_csv(A5_RUN/'runtime_roster.csv',dtype=str).fillna('')
required={'page_key','page','cohort','source_group','observed_path','seed_path'}
require(required<=set(roster.columns),f'Roster missing {sorted(required-set(roster.columns))}')
require(not roster.page_key.duplicated().any(),'Duplicate page_key')
RUN=OUTPUT_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_notebook84')
TABLES=RUN/'tables'; VISUALS=RUN/'visuals'; UPLOAD=RUN/'upload'
for p in [RUN,TABLES,VISUALS,UPLOAD]: p.mkdir(parents=True,exist_ok=True)
identity={'notebook':'84_A5_A1_Based_Safe_Cleanup_Evaluation','created_utc':datetime.now(timezone.utc).isoformat(),'a5_run':str(A5_RUN),'a5_lock_sha256':sha256_file(A5_RUN/'prediction_lock.json'),'margins_pp':{'exact_print_loss':MAX_EXACT_PRINT_PRESERVATION_LOSS_PP,'material_print_worsening':MAX_MATERIAL_PRINT_WORSENING_INCREASE_PP,'annotation_gain':MIN_ANNOTATION_IMPROVEMENT_GAIN_PP},'catastrophic_print_component_pixels':CATASTROPHIC_PRINT_COMPONENT_PIXELS,'promotion_authority':False}
(RUN/'experiment_identity.json').write_text(json.dumps(identity,indent=2)); print(json.dumps(identity,indent=2))

## 2. Validate A5 packages and enforce the A1 fallback contract

In [ ]:
rows=[]; exclusions=[]
for _,r in roster.iterrows():
    key=str(r.page_key); d=A5_RUN/'pages'/key
    files={'observed':'observed.png','seed':'seed.png','authority':'authority.png','ambiguous':'ambiguous.png','V89':'v8_9.png','A1':'a1.png','A3':'a3.png','A5':'a5.png','cleanup':'a5_cleanup_mask.png'}
    missing=[n for n,f in files.items() if not (d/f).exists()]
    if missing:
        exclusions.append({'page_key':key,'page':r.page,'cohort':r.cohort,'source_group':r.source_group,'reason':'missing:'+','.join(missing)}); continue
    a={n:read_gray(d/f) for n,f in files.items()}; shape=a['observed'].shape
    require(all(x.shape==shape for x in a.values()),f'{key}: shape mismatch')
    auth=a['authority']>0; amb=a['ambiguous']>0; cleanmask=a['cleanup']>0
    checks={'cleanup_outside_ambiguous':int(np.sum(cleanmask&~amb)),'ambiguous_outside_authority':int(np.sum(amb&~auth)),'a5_noncleanup_mismatch':int(np.sum(a['A5'][~cleanmask]!=a['A1'][~cleanmask])),'a5_outside_authority_drift':int(np.sum(a['A5'][~auth]!=a['observed'][~auth])),'a1_ambiguous_observed_mismatch':int(np.sum(a['A1'][amb]!=a['observed'][amb]))}
    require(all(v==0 for v in checks.values()),f'{key}: integrity failure {checks}')
    rows.append({'page_key':key,'page':str(r.page),'cohort':str(r.cohort),'source_group':str(r.source_group),'clean_path':str(r.clean_path) if 'clean_path' in roster.columns else '','authority_pixels':int(auth.sum()),'ambiguous_pixels':int(amb.sum()),'cleanup_pixels':int(cleanmask.sum()),'cleanup_fraction':float(cleanmask.sum()/max(amb.sum(),1)),'a5_vs_a1_pixels':int(np.sum(a['A5']!=a['A1'])),**checks})
inventory=pd.DataFrame(rows); exclusions=pd.DataFrame(exclusions)
require(len(inventory)>0,'No valid A5 pages')
inventory.to_csv(TABLES/'a5_page_inventory.csv',index=False); exclusions.to_csv(TABLES/'a5_input_exclusions.csv',index=False)
print('Valid:',len(inventory),'Excluded:',len(exclusions)); display(inventory.groupby('cohort').agg(pages=('page_key','nunique'),cleanup=('cleanup_pixels','sum'),ambiguous=('ambiguous_pixels','sum')).reset_index())

## 3. Resolve paired clean pages only after the A5 lock

In [ ]:
CLEAN_DIR=ROOT/'source'/'clean'; SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
clean_map={p.stem:p for p in CLEAN_DIR.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if CLEAN_DIR.exists() else {}
def resolve_clean(row):
    p=str(row.get('clean_path','') or '')
    if p and Path(p).exists(): return Path(p)
    for k in [str(row.page),str(row.page_key)]:
        if k in clean_map:return clean_map[k]
    return None
paired=inventory[inventory.cohort=='PAIRED_DEVELOPMENT'].copy(); paired['resolved_clean_path']=[str(resolve_clean(r)) if resolve_clean(r) else '' for _,r in paired.iterrows()]
missing=paired[paired.resolved_clean_path=='']; paired=paired[paired.resolved_clean_path!=''].copy()
require(len(paired)>0,'No paired truth pages'); paired.to_csv(TABLES/'paired_truth_roster.csv',index=False); missing.to_csv(TABLES/'paired_missing_clean.csv',index=False)
print('Paired truth:',len(paired),'Missing clean:',len(missing))

## 4. Score V8.9, A1, A3 and A5 against paired truth

In [ ]:
def printed_ink(clean): return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def profile(seed):
    m=seed>0
    if not m.any():return {'p90_depth':0.,'area_per_skeleton':0.}
    d=cv2.distanceTransform(m.astype(np.uint8),cv2.DIST_L2,5); sk=skeletonize(m)
    return {'p90_depth':float(np.quantile(d[m],.9)),'area_per_skeleton':float(m.sum()/max(sk.sum(),1))}
def score(out,obs,clean,auth,tp,ta,tu):
    oe=np.abs(obs.astype(np.int16)-clean.astype(np.int16)); ye=np.abs(out.astype(np.int16)-clean.astype(np.int16)); p=auth&tp; a=auth&ta; u=auth&tu
    q={'truth_print_pixels':int(p.sum()),'truth_annotation_pixels':int(a.sum()),'truth_uncertain_pixels':int(u.sum()),'print_exact_observed':int(np.sum(p&(out==obs))),'print_not_worsened':int(np.sum(p&(ye<=oe))),'print_materially_worsened':int(np.sum(p&(ye>=oe+MATERIAL_MARGIN))),'annotation_materially_improved':int(np.sum(a&(ye<=oe-MATERIAL_MARGIN))),'annotation_any_improved':int(np.sum(a&(ye<oe))),'annotation_left_observed':int(np.sum(a&(out==obs)))}
    for m,dn in [('print_exact_observed','truth_print_pixels'),('print_not_worsened','truth_print_pixels'),('print_materially_worsened','truth_print_pixels'),('annotation_materially_improved','truth_annotation_pixels'),('annotation_any_improved','truth_annotation_pixels'),('annotation_left_observed','truth_annotation_pixels')]:q[m+'_rate']=q[m]/max(q[dn],1)
    return q

scores=[]; comps=[]
for _,r in paired.iterrows():
    key=str(r.page_key);d=A5_RUN/'pages'/key;obs=read_gray(d/'observed.png');clean=read_gray(r.resolved_clean_path);seed=read_gray(d/'seed.png');auth=read_gray(d/'authority.png')>0;amb=read_gray(d/'ambiguous.png')>0;cm=read_gray(d/'a5_cleanup_mask.png')>0
    require(obs.shape==clean.shape==seed.shape,f'{key}: truth shape mismatch')
    cp=printed_ink(clean);k=2*PRINT_TOLERANCE_RADIUS+1;ct=cv2.dilate(cp.astype(np.uint8),np.ones((k,k),np.uint8))>0
    tp=auth&ct;ta=auth&(obs<DARK_THRESHOLD)&~ct;tu=auth&~tp&~ta;pr=profile(seed)
    outs={x:read_gray(d/f) for x,f in {'V89':'v8_9.png','A1':'a1.png','A3':'a3.png','A5':'a5.png'}.items()}
    for arm,out in outs.items():scores.append({'page_key':key,'page':r.page,'source_group':r.source_group,'arm':arm,'cleanup_pixels':int(cm.sum()),'ambiguous_pixels':int(amb.sum()),**pr,**score(out,obs,clean,auth,tp,ta,tu)})
    false=cm&tp&(outs['A5']!=outs['A1']);n,lab,st,_=cv2.connectedComponentsWithStats(false.astype(np.uint8),8)
    for cid in range(1,n):
        area=int(st[cid,cv2.CC_STAT_AREA]);comps.append({'page_key':key,'page':r.page,'source_group':r.source_group,'component_id':cid,'print_pixels':area,'catastrophic':area>=CATASTROPHIC_PRINT_COMPONENT_PIXELS,'x':int(st[cid,0]),'y':int(st[cid,1]),'w':int(st[cid,2]),'h':int(st[cid,3])})
    gc.collect()
page_scores=pd.DataFrame(scores);components=pd.DataFrame(comps)
page_scores.to_csv(TABLES/'paired_truth_by_page.csv',index=False);components.to_csv(TABLES/'a5_false_cleanup_components.csv',index=False)
print('Pages:',page_scores.page_key.nunique(),'Catastrophes:',int(components.catastrophic.sum()) if len(components) else 0)

## 5. Pooled summaries and page-clustered A5-versus-A1 deltas

In [ ]:
sumcols=['truth_print_pixels','truth_annotation_pixels','truth_uncertain_pixels','print_exact_observed','print_not_worsened','print_materially_worsened','annotation_materially_improved','annotation_any_improved','annotation_left_observed']
pooled=page_scores.groupby('arm',as_index=False).agg(pages=('page_key','nunique'),**{c:(c,'sum') for c in sumcols})
for m,dn in [('print_exact_observed','truth_print_pixels'),('print_not_worsened','truth_print_pixels'),('print_materially_worsened','truth_print_pixels'),('annotation_materially_improved','truth_annotation_pixels'),('annotation_any_improved','truth_annotation_pixels'),('annotation_left_observed','truth_annotation_pixels')]:pooled[m+'_rate']=pooled[m]/pooled[dn].clip(lower=1)
pooled.to_csv(TABLES/'pooled_arm_summary.csv',index=False);display(pooled)

def boot(v,seed):
    v=np.asarray(v,float);v=v[np.isfinite(v)];rng=np.random.default_rng(seed);b=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(BOOTSTRAP_RESAMPLES)])
    return {'pages':len(v),'mean_delta':v.mean(),'median_delta':np.median(v),'ci95_low':np.quantile(b,.025),'ci95_high':np.quantile(b,.975),'wins':int((v>0).sum()),'losses':int((v<0).sum()),'ties':int((v==0).sum())}
metrics={'print_exact_observed_rate':('Exact print preservation','benefit'),'print_not_worsened_rate':('Print not worsened','benefit'),'print_materially_worsened_rate':('Material print worsening','harm'),'annotation_materially_improved_rate':('Annotation materially improved','benefit'),'annotation_any_improved_rate':('Annotation any improvement','benefit')}
base=page_scores[page_scores.arm.isin(['A1','A5'])].pivot(index=['page_key','page','source_group'],columns='arm',values=list(metrics)).dropna();rows=[]
for m,(label,direction) in metrics.items():
    raw=base[(m,'A5')]-base[(m,'A1')];benefit=raw if direction=='benefit' else -raw;rows.append({'metric':m,'label':label,'direction':direction,'raw_A5_minus_A1_mean':raw.mean(),**boot(benefit,BOOTSTRAP_SEED+sum(map(ord,m)))})
summary=pd.DataFrame(rows);summary.to_csv(TABLES/'A5_vs_A1_page_clustered_bootstrap.csv',index=False);display(summary)
lookup=summary.set_index('metric');exact_low=100*lookup.loc['print_exact_observed_rate','ci95_low'];worsen_upper=-100*lookup.loc['print_materially_worsened_rate','ci95_low'];ann_low=100*lookup.loc['annotation_materially_improved_rate','ci95_low'];ann_mean=100*lookup.loc['annotation_materially_improved_rate','mean_delta']

## 6. Page/source/mask-depth stability and visual review queue

In [ ]:
flat=base.reset_index();flat.columns=['__'.join(str(x) for x in c if str(x)) if isinstance(c,tuple) else str(c) for c in flat.columns]
for m,(label,direction) in metrics.items():
    raw=flat[f'{m}__A5']-flat[f'{m}__A1'];flat[f'benefit__{m}']=raw if direction=='benefit' else -raw
flat.to_csv(TABLES/'A5_vs_A1_page_deltas.csv',index=False)
source=[]
for sg,g in flat.groupby('source_group'):
    for m,(label,direction) in metrics.items():
        x=g[f'benefit__{m}'].to_numpy(float);source.append({'source_group':sg,'pages':len(g),'metric':m,'mean_benefit':x.mean(),'wins':int((x>0).sum()),'losses':int((x<0).sum()),'ties':int((x==0).sum())})
source=pd.DataFrame(source);source.to_csv(TABLES/'A5_vs_A1_by_source_group.csv',index=False)
prof=page_scores[page_scores.arm=='A1'][['page_key','p90_depth']].drop_duplicates();prof['depth_band']=pd.qcut(prof.p90_depth.rank(method='first'),4,labels=['Q1_THIN','Q2','Q3','Q4_BROAD'])
pdeltas=flat.merge(prof,on='page_key');pr=[]
for b,g in pdeltas.groupby('depth_band',observed=True):
    for m in metrics:
        x=g[f'benefit__{m}'].to_numpy(float);pr.append({'depth_band':str(b),'pages':len(g),'metric':m,'mean_benefit':x.mean(),'wins':int((x>0).sum()),'losses':int((x<0).sum()),'ties':int((x==0).sum())})
profile_summary=pd.DataFrame(pr);profile_summary.to_csv(TABLES/'A5_vs_A1_by_mask_depth_band.csv',index=False)
flat['a5_vs_a1_pixels']=flat.page_key.map(inventory.set_index('page_key').a5_vs_a1_pixels);flat['print_harm']=flat['print_materially_worsened_rate__A5']-flat['print_materially_worsened_rate__A1'];flat['annotation_gain']=flat['annotation_materially_improved_rate__A5']-flat['annotation_materially_improved_rate__A1'];flat.sort_values(['print_harm','a5_vs_a1_pixels'],ascending=False).to_csv(TABLES/'visual_review_queue.csv',index=False)
plt.figure(figsize=(8,6));plt.axhline(0,color='grey');plt.axvline(0,color='grey');plt.scatter(100*flat.annotation_gain,-100*flat.print_harm);plt.xlabel('Annotation gain vs A1 (pp)');plt.ylabel('Reduction in print worsening vs A1 (pp)');plt.tight_layout();plt.savefig(VISUALS/'A5_vs_A1_tradeoff.png',dpi=180);plt.close()

## 7. Final fail-closed verdict and bundle

In [ ]:
source_print=source[(source.metric=='print_materially_worsened_rate')&(source.pages>=3)]
profile_print=profile_summary[profile_summary.metric=='print_materially_worsened_rate']
mandatory={'A1_fallback_integrity':bool((inventory.a5_noncleanup_mismatch==0).all()),'outside_authority_immutability':bool((inventory.a5_outside_authority_drift==0).all()),'exact_print_noninferiority':bool(exact_low>=-MAX_EXACT_PRINT_PRESERVATION_LOSS_PP),'material_print_worsening_noninferiority':bool(worsen_upper<=MAX_MATERIAL_PRINT_WORSENING_INCREASE_PP),'annotation_gain':bool(ann_low>0 and ann_mean>=MIN_ANNOTATION_IMPROVEMENT_GAIN_PP),'zero_catastrophes':bool((int(components.catastrophic.sum()) if len(components) else 0)==0),'source_stability':bool((source_print.mean_benefit>=0).all()),'mask_depth_stability':bool((profile_print.mean_benefit>=0).all())}
if all(mandatory.values()): verdict='A5_DEVELOPMENT_GATES_PASS_FREEZE_CHALLENGER_ONLY'
elif mandatory['annotation_gain'] and not (mandatory['exact_print_noninferiority'] and mandatory['material_print_worsening_noninferiority'] and mandatory['zero_catastrophes']):verdict='A5_REJECT_PRINT_SAFETY_FAILURE'
elif not mandatory['annotation_gain']:verdict='A5_LOW_OR_UNCERTAIN_CLEANUP_BENEFIT'
else:verdict='A5_REJECT_STABILITY_OR_INTEGRITY_FAILURE'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'a5_run':str(A5_RUN),'mandatory_gates':mandatory,'paired_pages':int(page_scores.page_key.nunique()),'valid_pages':len(inventory),'excluded_pages':len(exclusions),'catastrophic_components':int(components.catastrophic.sum()) if len(components) else 0,'exact_print_lower_bound_pp':float(exact_low),'material_worsening_harm_upper_bound_pp':float(worsen_upper),'annotation_gain_mean_pp':float(ann_mean),'annotation_gain_lower_bound_pp':float(ann_low),'promotion_authority':False}
(RUN/'notebook84_verdict.json').write_text(json.dumps(record,indent=2))
lines=['NOTEBOOK 84 — A5 EVALUATION','',f'Verdict: {verdict}',f'A5 run: {A5_RUN}',f'Valid pages: {len(inventory)}',f'Paired pages: {page_scores.page_key.nunique()}',f'Exact-print lower bound: {exact_low:.4f} pp',f'Print-worsening harmful upper bound: {worsen_upper:.4f} pp',f'Annotation gain mean/lower bound: {ann_mean:.4f}/{ann_low:.4f} pp',f'Catastrophes: {record["catastrophic_components"]}','','Mandatory gates:']+[f'  {k}: {v}' for k,v in mandatory.items()]+['','No production promotion is authorised.']
(RUN/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))
artefacts=[RUN/'executive_summary.txt',RUN/'notebook84_verdict.json',RUN/'experiment_identity.json',TABLES/'a5_page_inventory.csv',TABLES/'a5_input_exclusions.csv',TABLES/'paired_truth_by_page.csv',TABLES/'pooled_arm_summary.csv',TABLES/'A5_vs_A1_page_clustered_bootstrap.csv',TABLES/'A5_vs_A1_page_deltas.csv',TABLES/'A5_vs_A1_by_source_group.csv',TABLES/'A5_vs_A1_by_mask_depth_band.csv',TABLES/'a5_false_cleanup_components.csv',TABLES/'visual_review_queue.csv',VISUALS/'A5_vs_A1_tradeoff.png']
for p in artefacts:require(p.exists(),f'Missing {p}');shutil.copy2(p,UPLOAD/p.name)
manifest=[{'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in UPLOAD.iterdir() if p.is_file()];pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False);shutil.make_archive(str(RUN/'notebook84_a5_evaluation_bundle'),'zip',UPLOAD);print('Bundle:',str(RUN/'notebook84_a5_evaluation_bundle.zip'))